# Scenario

You are handed a live tracking feed of fuel levels across 50 distribution depots. Your mission is to isolate structural vulnerabilities before they trigger an operational shutdown.

Specifically, you need to filter the master sheet down to depots where stock is currently below 20% capacity AND the last telemetry update was more than 4 hours ago (indicating a possible sensor freeze or communication drop).

# Solution

```python
import pandas as pd 

# Load the operational asset data
df = pd.read_csv('depot_inventory.csv') 

# Apply Boolean Filters: Low stock AND stale data
# Note the use of bitwise '&' and parentheses around each explicit condition!
critical_conditions = (df['stock_percent'] < 20) & (df['hours_since_update'] > 4)
critical_depots = df[critical_conditions]

# Trigger operational alert system
print(f"🚨 ALERT: {len(critical_depots)} depots require immediate dispatch attention!")
print(critical_depots[['depot_id', 'location', 'stock_percent']])

```

# Scenario Two

You receive a temperature log from a piece of heavy equipment. The dataset is noisy: some rows contain standard NaN missing values, while others contain 999.9—a legacy hardcoded hardware error code indicating a temporary telemetry drop.

Your goal is to clean this log so it can be safely used by the engineering team.

In [2]:
# Solution

import numpy as np
import pandas as pd

# Load the noisy log (Simulated DataFrame)
df = pd.DataFrame({
    'timestamp': pd.date_range(start='2026-06-13 08:00', periods=6, freq='min'),
    'temp_celsius': [22.4, 22.5, 999.9, np.nan, 22.7, 150.0]  # Note the error code & impossible spike
})

print("Raw, Noisy Data:\n", df)
print("-" * 40)

# Replace hardware error codes (999.9) with official NaN values
df['temp_celsius'] = df['temp_celsius'].replace(999.9, np.nan)

# Forward-fill missing values (assuming equipment temp doesn't change instantly)
df['temp_celsius'] = df['temp_celsius'].ffill()

# Clip extreme outliers that are physically impossible for this machine (e.g., max 50°C)
# Anything above 50.0 will be capped down to 50.0 to strip out system spikes
df['temp_celsius'] = df['temp_celsius'].clip(upper=50.0)

print("Cleaned Operational Data:\n", df)

Raw, Noisy Data:
             timestamp  temp_celsius
0 2026-06-13 08:00:00          22.4
1 2026-06-13 08:01:00          22.5
2 2026-06-13 08:02:00         999.9
3 2026-06-13 08:03:00           NaN
4 2026-06-13 08:04:00          22.7
5 2026-06-13 08:05:00         150.0
----------------------------------------
Cleaned Operational Data:
             timestamp  temp_celsius
0 2026-06-13 08:00:00          22.4
1 2026-06-13 08:01:00          22.5
2 2026-06-13 08:02:00          22.5
3 2026-06-13 08:03:00          22.5
4 2026-06-13 08:04:00          22.7
5 2026-06-13 08:05:00          50.0


In [3]:
# 1. Setup simulated transactional log
np.random.seed(42)
data = {
    'branch': np.random.choice(['Nairobi', 'Mombasa', 'Kisumu'], size=300),
    'shift': np.random.choice(['Morning', 'Afternoon', 'Night'], size=300),
    'transactions_completed': np.random.randint(50, 250, size=300)
}
df = pd.DataFrame(data)

# Inject an intentional operational bottleneck (Night shift in Kisumu underperforms)
df.loc[(df['branch'] == 'Kisumu') & (df['shift'] == 'Night'), 'transactions_completed'] //= 3

# 2. Generate a Multi-Aggregation Summary
print("Branch & Shift Aggregation Summary:")
summary = df.groupby(['branch', 'shift'])['transactions_completed'].agg(['mean', 'max', 'count'])
print(summary.head(6))
print("-" * 50)

# 3. Pivot the data into an Executive Heatmap View
print("Pivot Table View (Average Transactions):")
pivot_report = pd.pivot_table(
    df, 
    values='transactions_completed', 
    index='branch', 
    columns='shift', 
    aggfunc='mean'
)
print(pivot_report)

Branch & Shift Aggregation Summary:
                         mean  max  count
branch  shift                            
Kisumu  Afternoon  143.789474  228     38
        Morning    159.600000  249     40
        Night       50.655172   78     29
Mombasa Afternoon  143.636364  249     33
        Morning    144.424242  243     33
        Night      158.214286  246     28
--------------------------------------------------
Pivot Table View (Average Transactions):
shift     Afternoon     Morning       Night
branch                                     
Kisumu   143.789474  159.600000   50.655172
Mombasa  143.636364  144.424242  158.214286
Nairobi  153.000000  156.100000  164.937500


# Scenario Three

You are monitoring a high-frequency pipeline pressure sensor that logs data every single second. The raw stream is incredibly noisy due to fluid turbulence, making it impossible to see if the pipe is degrading.

Your objective is to resample the stream into 1-minute averages, then apply a 1-hour rolling mean to filter the noise and check for a slow, downward pressure drift—a classic indicator of an upstream structural blockage.

In [1]:
# Solution

import numpy as np
import pandas as pd

# Simulate 2 hours of noisy, high-frequency data (1 reading per second)
time_range = pd.date_range(start='2026-06-13 12:00:00', periods=7200, freq='s')

# Vectorized generation: Create a steady downward drift hidden under heavy random noise
drift = np.linspace(120, 105, num=7200) 
noise = np.random.normal(0, 15, size=7200)
pressure_readings = drift + noise

df = pd.DataFrame({'pressure_psi': pressure_readings}, index=time_range)
print("Raw High-Frequency Telemetry (Every Second):")
print(df.head(3))
print("-" * 55)

# Resample raw seconds data into structured 1-Minute averages
minute_df = df.resample('1min').mean()
print("Resampled Data (1-Minute Averages):")
print(minute_df.head(3))
print("-" * 55)

# Apply a 1-Hour Rolling Mean (60-minute window) to expose the baseline trend
# min_periods=1 ensures we get values immediately instead of waiting for 60 empty rows
minute_df['rolling_trend_60m'] = minute_df['pressure_psi'].rolling(window=60, min_periods=1).mean()

print("Final Smoothed Pipeline Trend Data:")
print(minute_df[['pressure_psi', 'rolling_trend_60m']].tail(3))

Raw High-Frequency Telemetry (Every Second):
                     pressure_psi
2026-06-13 12:00:00    129.191464
2026-06-13 12:00:01    142.321045
2026-06-13 12:00:02    126.716456
-------------------------------------------------------
Resampled Data (1-Minute Averages):
                     pressure_psi
2026-06-13 12:00:00    121.068427
2026-06-13 12:01:00    120.042595
2026-06-13 12:02:00    117.405711
-------------------------------------------------------
Final Smoothed Pipeline Trend Data:
                     pressure_psi  rolling_trend_60m
2026-06-13 13:57:00    109.328147         109.026011
2026-06-13 13:58:00    104.557933         108.842529
2026-06-13 13:59:00    102.168029         108.660651
